# 08a — final morph cluster assignments

**Feeds:** Supplementary Data 1

**Position in the chain:** run the numbered stages in order

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up. `PROJECT_ROOT`, which was the analysis directory, is now `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, in the same layout. `DEV_ROOT`, under which the notebook writes, is `$SCRNASEQ_RESULTS_ROOT/trunk_main_dev/` (default `scrnaseq/output/trunk_main_dev/`) instead of the analysis directory's `trunk_main_dev/`.
2. Removed the `.to_clipboard()` call, which copied a table to the system clipboard for pasting into the Supplementary Data spreadsheet and fail on a machine without a clipboard. The expression before each call is unchanged.

No other line of code was changed.


# 08a - Final Morph Cluster Assignments


Depends on: 02_trunk_main, 03_lpm_subclustering, 04_somite_subclustering, 05_fbmb_subclustering, 06_rp_nc_subclustering.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)

import os
os.chdir(REPO_ROOT)


## Setup and Imports


In [ ]:
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:
from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under trunk_main_dev/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "trunk_main_dev"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
from src.trunk_morph_ref.aggregation import (
    cluster_averages_sparse_safe,
)
from src.trunk_morph_ref.correlation import (
    gene_corrcoef_sparse_safe,
)
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    contains_symbol,
    ensure_gene_id_index,
    filter_genes_sparse_safe as filter_genes,
    install_scanpy_symbol_defaults,
    normalize_unit_variance_sparse_safe as normalize_unit_variance,
    resolve_symbol_dict,
    resolve_symbols,
    symbols_missing,
    symbols_present,
    var_names_to_symbols,
)

# Use gene symbols for plot labels while keeping `var_names` as stable gene IDs.
install_scanpy_symbol_defaults(sc)


In [ ]:
from src.trunk_morph_ref.pipeline_io import (
    load_h5ad,
    load_npy,
    load_pickle,
    save_h5ad,
    save_json,
    save_npy,
    save_pickle,
    stage_dir,
)

In [ ]:
trunk_path = stage_dir(RESULTS_DIR, "02_trunk_main")
lpm_path = stage_dir(RESULTS_DIR, "03_lpm_subclustering")
somite_path = stage_dir(RESULTS_DIR, "04_somite_subclustering")
fbmb_path = stage_dir(RESULTS_DIR, "05_fbmb_subclustering")
rpnc_path = stage_dir(RESULTS_DIR, "06_rp_nc_subclustering")

adata_morph_SMD_ = load_h5ad(trunk_path / "adata_morph_SMD_.h5ad")
adata_morph_lpmendo_SMD = load_h5ad(lpm_path / "adata_morph_lpmendo_SMD.h5ad")
adata_morph_fbmb_ = load_h5ad(fbmb_path / "adata_morph_fbmb_.h5ad")
adata_morph_nc_ = load_h5ad(rpnc_path / "adata_morph_nc_.h5ad")
adata_morph_rp_ = load_h5ad(rpnc_path / "adata_morph_rp_.h5ad")
somite_subcluster_labels = pd.read_csv(
    somite_path / "somite_subcluster_labels.csv", index_col=0
)

for _adata in [
    adata_morph_SMD_,
    adata_morph_lpmendo_SMD,
    adata_morph_fbmb_,
    adata_morph_nc_,
    adata_morph_rp_,
]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)

print("Loaded upstream intermediates for final morph cluster assignments")


## Supp. Data 1 - Final Cluster Assignments


In [ ]:
for cell_id in adata_morph_lpmendo_SMD.obs_names:
    adata_morph_SMD_.obs.loc[cell_id, "leiden_morph_sub"] = adata_morph_lpmendo_SMD.obs.loc[
        cell_id, "leiden_morph_lpm"
    ]

for cell_id in adata_morph_nc_.obs_names:
    adata_morph_SMD_.obs.loc[cell_id, "leiden_morph_sub"] = adata_morph_nc_.obs.loc[
        cell_id, "leiden_morph_nc"
    ]
for cell_id in adata_morph_rp_.obs_names:
    adata_morph_SMD_.obs.loc[cell_id, "leiden_morph_sub"] = adata_morph_rp_.obs.loc[
        cell_id, "leiden_morph_rp"
    ]
for cell_id in somite_subcluster_labels.index:
    adata_morph_SMD_.obs.loc[cell_id, "leiden_morph_sub"] = somite_subcluster_labels.loc[
        cell_id, "leiden_morph_somite"
    ]
for cell_id in adata_morph_fbmb_.obs_names:
    adata_morph_SMD_.obs.loc[cell_id, "leiden_morph_sub"] = adata_morph_fbmb_.obs.loc[
        cell_id, "leiden_morph_fbmb"
    ]

In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 1 - "Gene markers and differential expression supporting trunk morph cell type annotations"
# Sheet 3 "Cluster Assignments"

adata_morph_SMD_.obs[["source", "leiden_morph", "leiden_morph_sub"]].sort_values(
    "leiden_morph"
)

## Save Stage Outputs
Persist final trunk morph cluster assignments with integrated subcluster labels.


In [ ]:
stage_path = stage_dir(RESULTS_DIR, "08a_final_morph_cluster_assignments")

cluster_assignment_cols = ["source", "leiden_morph", "leiden_morph_sub"]
cluster_assignments = adata_morph_SMD_.obs[cluster_assignment_cols].sort_values("leiden_morph")
cluster_assignments.to_csv(stage_path / "morph_cluster_assignments_with_subclusters.csv")

ensure_gene_id_index(adata_morph_SMD_)
assert_gene_id_index(adata_morph_SMD_)
save_h5ad(adata_morph_SMD_, stage_path / "adata_morph_SMD_with_subclusters.h5ad")

save_json(
    {"stage": "08a_final_morph_cluster_assignments", "seed_policy": "all seeds set to 0"},
    stage_path / "meta.json",
)

print(f"Saved final morph cluster assignments to {stage_path}")
